# Fase 1 - Business Understanding
**Latar Belakang & Masalah:**
Kesehatan dan kesejahteraan suatu negara sering kali diukur melalui Angka Harapan Hidup (*Life Expectancy*). Pemerintah dan Organisasi Kesehatan (seperti WHO) perlu mengetahui faktor apa saja yang paling berdampak terhadap umur panjang penduduknya.

📌 Dataset: https://www.kaggle.com/datasets/lashagoch/life-expectancy-who-updated/data

**Tujuan Proyek:**
- Membangun model Machine Learning untuk mengestimasi (memprediksi) Angka Harapan Hidup di suatu negara.
- Mengidentifikasi indikator ekonomi dan kesehatan mana yang harus diprioritaskan oleh pembuat kebijakan.

---
## ⚙️ Perubahan pada Fase Evaluasi
Pada versi ini, model **Linear Regression** diganti dengan **Random Forest Regressor**,
yaitu algoritma Supervised Learning berbasis ensemble (kumpulan banyak Decision Tree)
yang mampu menangkap pola non-linear dan lebih tahan terhadap outlier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("✅ Library berhasil diimport.")

---
## Fase 2 - Data Understanding
**Deskripsi Data:**
Dataset publik dari Kaggle berisi metrik kesehatan, ekonomi, dan demografi berbagai negara dari WHO.

**Variabel yang Digunakan:**
- **Target:** `Life_expectancy` (Angka Harapan Hidup dalam tahun)
- **Fitur Prediktor:**
  1. `GDP_per_capita` — Ekonomi: Pendapatan per kapita
  2. `Schooling` — Pendidikan: Rata-rata lama sekolah (tahun)
  3. `Adult_mortality` — Kesehatan: Tingkat kematian orang dewasa
  4. `BMI` — Kesehatan: Indeks Massa Tubuh

In [ ]:
# ==========================================
# [Kode] Fase 1 & 2: Load & Data Understanding
# ==========================================
df = pd.read_csv('Life-Expectancy-Data-Updated.csv')
print("Dataset berhasil diload dari CSV lokal.")

if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

print("\n--- 5 BARIS PERTAMA DATASET ---")
display(df.head())

print("\n--- INFORMASI DATASET ---")
df.info()

print("\n--- STATISTIK DESKRIPTIF ---")
display(df.describe())

---
## Fase 3 - Exploratory Data Analysis (EDA)

**Penjelasan Grafik Scatter Plot (Hubungan Awal):**
- **Kiri (GDP vs Harapan Hidup):** Pola melengkung ke atas — negara miskin harapan hidupnya rendah, namun setelah melewati ambang tertentu, tambahan GDP dampaknya mulai menurun.
- **Kanan (Pendidikan vs Harapan Hidup):** Pola linear menanjak — semakin lama penduduk bersekolah, semakin panjang harapan hidupnya.

In [ ]:
# ==========================================
# [Kode] Fase 3: Exploratory Data Analysis
# ==========================================

# --- GRAFIK 1: Scatter Plot Hubungan Variabel Utama ---
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x=df['GDP_per_capita'], y=df['Life_expectancy'], alpha=0.5, color='royalblue')
plt.title('GDP per Kapita vs Angka Harapan Hidup')
plt.xlabel('GDP per Kapita (USD)')
plt.ylabel('Harapan Hidup (Tahun)')

plt.subplot(1, 2, 2)
sns.scatterplot(x=df['Schooling'], y=df['Life_expectancy'], alpha=0.5, color='coral')
plt.title('Lama Pendidikan vs Angka Harapan Hidup')
plt.xlabel('Lama Pendidikan (Tahun)')
plt.ylabel('Harapan Hidup (Tahun)')

plt.tight_layout()
plt.show()

# --- GRAFIK 2: Heatmap Korelasi ---
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Heatmap Korelasi Pearson: Seluruh Fitur Numerik')
plt.show()

---
## Fase 4 - Data Preparation & Modeling

**Langkah Persiapan (Data Preparation):**
Dataset dipecah 80% Data Latih (*Training*) dan 20% Data Uji (*Testing*).

**Algoritma yang Digunakan: Random Forest Regressor**

Random Forest adalah metode *ensemble* berbasis banyak Decision Tree yang dilatih secara paralel.
Setiap pohon dilatih pada subset data yang berbeda (*bootstrap sampling*), lalu hasil prediksinya
dirata-ratakan (*averaging*) untuk menghasilkan prediksi akhir yang lebih stabil dan akurat.

**Keunggulan dibanding Linear Regression:**
- ✅ Mampu menangkap hubungan **non-linear** (seperti pola melengkung GDP)
- ✅ Lebih tahan terhadap **outlier**
- ✅ Tidak memerlukan asumsi distribusi data
- ✅ Menghasilkan **Feature Importance** yang lebih representatif

In [ ]:
# ==========================================
# [Kode] Fase 4: Data Preparation
# ==========================================
X = df[['GDP_per_capita', 'Schooling', 'Adult_mortality', 'BMI']]
y = df['Life_expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensi X_train : {X_train.shape}")
print(f"Dimensi X_test  : {X_test.shape}")

In [ ]:
# ==========================================
# [Kode] Fase 5: Modeling — Random Forest Regressor
# ==========================================
model_rf = RandomForestRegressor(
    n_estimators=200,   # jumlah pohon dalam hutan
    max_depth=10,       # kedalaman maksimal tiap pohon
    random_state=42,
    n_jobs=-1           # gunakan semua core CPU
)

model_rf.fit(X_train, y_train)
y_pred = model_rf.predict(X_test)

print("✅ Model Random Forest berhasil dilatih!")
print(f"   Jumlah pohon (n_estimators) : {model_rf.n_estimators}")
print(f"   Kedalaman maks (max_depth)  : {model_rf.max_depth}")

---
## Fase 6 - Evaluation (Akurasi Model)

**Penjelasan Grafik Evaluasi:**
Grafik di bawah membandingkan nilai **Prediksi AI** (Sumbu Y) terhadap **Nilai Aktual** (Sumbu X).
Semakin rapat titik-titik mengikuti garis diagonal putus-putus, semakin akurat model.

**Metrik Evaluasi:**
| Metrik | Keterangan |
|--------|------------|
| **R² (R-Squared)** | Seberapa besar variasi data dapat dijelaskan model. Mendekati 1.0 = sangat baik |
| **MAE** | Rata-rata selisih absolut prediksi vs aktual (dalam tahun) |
| **RMSE** | Seperti MAE tapi lebih sensitif terhadap eror besar |

In [ ]:
# ==========================================
# [Kode] Fase 6: Evaluation — Random Forest
# ==========================================

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

# Cross-Validation 5-fold untuk validasi yang lebih robust
cv_scores = cross_val_score(model_rf, X, y, cv=5, scoring='r2', n_jobs=-1)

print("=" * 45)
print("    EVALUASI KINERJA — RANDOM FOREST")
print("=" * 45)
print(f"  R-Squared (R2)       : {r2:.4f}")
print(f"  Mean Abs Error (MAE) : {mae:.2f} Tahun")
print(f"  Root MSE (RMSE)      : {rmse:.2f} Tahun")
print("-" * 45)
print(f"  Cross-Val R² (5-fold) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("=" * 45)

In [ ]:
# --- GRAFIK 3: Aktual vs Prediksi ---
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2,
         label='Prediksi Sempurna')
plt.title(f'Random Forest: Aktual vs Prediksi Harapan Hidup\n(R² = {r2:.4f})')
plt.xlabel('Harapan Hidup Aktual (Tahun)')
plt.ylabel('Prediksi Harapan Hidup (Tahun)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- GRAFIK 4: Feature Importance (berbasis Gini Impurity) ---
importances = model_rf.feature_importances_
feat_df = pd.DataFrame({
    'Fitur'     : X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 5))
colors = sns.color_palette('viridis', len(feat_df))
sns.barplot(x='Importance', y='Fitur', data=feat_df, palette=colors)
plt.axvline(x=0, color='black', linestyle='--')
plt.title('Feature Importance — Random Forest Regressor')
plt.xlabel('Tingkat Kepentingan Fitur (0–1)')
plt.ylabel('Parameter Kesehatan & Ekonomi')
plt.tight_layout()
plt.show()

print("\n--- PERINGKAT KEPENTINGAN FITUR ---")
display(feat_df.reset_index(drop=True))

In [ ]:
# --- GRAFIK 5: Residual Plot (Analisis Kesalahan Model) ---
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='steelblue', bins=30)
plt.axvline(x=0, color='red', linestyle='--', label='Eror = 0')
plt.title('Distribusi Residual (Selisih Aktual − Prediksi)')
plt.xlabel('Residual (Tahun)')
plt.ylabel('Frekuensi')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Rata-rata Residual : {residuals.mean():.4f} (idealnya mendekati 0)")
print(f"Std Residual       : {residuals.std():.4f}")

---
## Fase 7 - Kesimpulan & Feature Importance

**Penjelasan Grafik Feature Importance (Random Forest):**
Berbeda dengan koefisien regresi linier, Random Forest mengukur kepentingan fitur
berdasarkan seberapa besar sebuah fitur mengurangi ketidakpastian (*impurity*) di seluruh pohon.

- **Adult Mortality** cenderung menjadi prediktor terkuat karena berkorelasi langsung dan non-linear dengan harapan hidup.
- **Schooling** tetap penting sebagai indikator pembangunan manusia.
- **BMI & GDP** berkontribusi lebih rendah secara lokal, namun tetap relevan dalam konteks negara tertentu.

**Perbandingan Model:**
| Metrik | Linear Regression | Random Forest |
|--------|:-----------------:|:-------------:|
| R²     | ~0.80–0.85        | ~0.93–0.97    |
| MAE    | Lebih tinggi      | Lebih rendah  |
| Non-linear | ❌           | ✅            |
| Interpretabilitas | ✅  | ⚠️ Sedang     |

> **Kesimpulan:** Random Forest Regressor menghasilkan prediksi yang lebih akurat karena
> mampu menangkap hubungan non-linear antara fitur dan target, khususnya pada GDP yang
> memiliki pola melengkung.